# EZStats - Train Player-Centric Event Spotter (SN-PCBAS-2026)
48 training matches, 8 action classes + team prediction head.
Run order: A -> B -> C -> D (READ output!) -> E -> F -> G -> H -> I

## Cell A - Install + Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
subprocess.run(['apt-get', 'install', '-y', '-q', 'p7zip-full'], check=True)
subprocess.run(['pip', 'install', '-q', 'huggingface_hub', 'datasets', 'torch', 'torchvision', 'h5py'], check=True)

from pathlib import Path
PCBAS_ROOT    = Path('/content/drive/MyDrive/ezstats/sn-pcbas-2026')
LOCAL_EXTRACT = Path('/content/pcbas-extracted')
FEAT_DIR      = Path('/content/drive/MyDrive/ezstats/pcbas-features')
for p in [PCBAS_ROOT, LOCAL_EXTRACT, FEAT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print('Ready.')

## Cell B - Login + Download
Downloads only 352x640 videos (~24 GB total). ResNet18 resizes to 224x224 anyway so Full HD gives no accuracy benefit.
`notebook_login()` will open a widget -- create a token at huggingface.co/settings/tokens if prompted.

In [ ]:
from huggingface_hub import notebook_login, hf_hub_download
from pathlib import Path

PCBAS_ROOT = Path('/content/drive/MyDrive/ezstats/sn-pcbas-2026')
PCBAS_ROOT.mkdir(parents=True, exist_ok=True)

notebook_login()

REPO  = 'SoccerNet/SN-PCBAS-2026'
FILES = [
    ('videos_352x640_TRAIN.zip', '22.8 GB'),
    ('videos_352x640_VAL.zip',   '1.4 GB'),
    ('tactical_data_TRAIN.zip',  '2.5 GB'),
    ('tactical_data_VAL.zip',    '157 MB'),
    ('tactical_data_format.txt', 'tiny'),
]

for fname, size in FILES:
    local = PCBAS_ROOT / fname
    if local.exists():
        print(f'SKIP: {fname}')
        continue
    print(f'Downloading {fname} ({size})...')
    hf_hub_download(
        repo_id=REPO,
        repo_type='dataset',
        filename=fname,
        local_dir=str(PCBAS_ROOT),
    )
    print(f'  Done.')

print('All files on Drive.')

## Cell C - Extract + Print Format Spec
Tries standard SoccerNet password `s0cc3rn3t`. If wrong, the NDA form at soccer-net.org gives the PCBAS-2026 key.
Prints `tactical_data_format.txt` so you know the label structure before Cell D.

In [ ]:
import subprocess
from pathlib import Path

PCBAS_ROOT    = Path('/content/drive/MyDrive/ezstats/sn-pcbas-2026')
LOCAL_EXTRACT = Path('/content/pcbas-extracted')
LOCAL_EXTRACT.mkdir(exist_ok=True)

PASSWORD = 's0cc3rn3t'

for zip_name in ['tactical_data_TRAIN.zip', 'tactical_data_VAL.zip',
                 'videos_352x640_TRAIN.zip', 'videos_352x640_VAL.zip']:
    zf = PCBAS_ROOT / zip_name
    if not zf.exists():
        print(f'SKIP (not downloaded): {zip_name}')
        continue
    size_gb = zf.stat().st_size / 1024**3
    print(f'Extracting {zip_name} ({size_gb:.1f} GB)...')
    result = subprocess.run(
        ['7z', 'x', f'-p{PASSWORD}', f'-o{LOCAL_EXTRACT}', '-y', str(zf)],
        capture_output=True, text=True
    )
    if result.returncode > 1:
        print(f'  ERROR (code {result.returncode}):')
        print(result.stdout[-500:])
        print('  If Wrong Password -- submit NDA form at soccer-net.org for PCBAS-2026 key.')
    else:
        print(f'  Done.')

fmt = PCBAS_ROOT / 'tactical_data_format.txt'
if fmt.exists():
    print('\n=== tactical_data_format.txt ===')
    print(fmt.read_text())

labels = sorted(LOCAL_EXTRACT.rglob('*.json'))
videos = sorted(LOCAL_EXTRACT.rglob('*.mp4')) + sorted(LOCAL_EXTRACT.rglob('*.mkv'))
print(f'\n{len(labels)} label files, {len(videos)} videos')
for l in labels[:6]: print(' L:', l.relative_to(LOCAL_EXTRACT))
for v in videos[:6]: print(' V:', v.relative_to(LOCAL_EXTRACT))

## Cell D - Inspect Label Format (CRITICAL - read output before running F)
Determines JSON structure, exact class name strings, position unit (frames vs ms), and team field format.
Update BALL_ACTION_CLASSES and get_pos() in Cell F based on what you see here.

In [ ]:
import h5py
import numpy as np
from pathlib import Path
from collections import Counter

LOCAL_EXTRACT = Path('/content/pcbas-extracted')

train_h5 = LOCAL_EXTRACT / 'train_tactical_data.h5'
val_h5   = LOCAL_EXTRACT / 'val_tactical_data.h5'

for h5_path in [train_h5, val_h5]:
    if not h5_path.exists():
        print(f'NOT FOUND: {h5_path.name}')
        continue
    print(f'\n=== {h5_path.name} ===')
    with h5py.File(str(h5_path), 'r') as f:
        keys = sorted(f.keys())
        print(f'  Games (keys): {len(keys)} -- first 10: {keys[:10]}')

        # Inspect first game
        first_key = keys[0]
        grp = f[first_key]
        print(f'\n  [{first_key}] type: {type(grp)}')
        if isinstance(grp, h5py.Dataset):
            data = grp[:]
            print(f'  Shape: {data.shape}, dtype: {data.dtype}')
            print(f'  First 3 rows:\n{data[:3]}')
            if data.ndim == 2 and data.shape[1] >= 14:
                frames_col  = data[:, 0].astype(int)
                classes_col = data[:, 13].astype(int)
                teams_col   = data[:, 2].astype(int)
                print(f'  Frame range: {frames_col.min()} - {frames_col.max()}')
                print(f'  Unique class values (col 13):', sorted(Counter(classes_col.tolist()).items()))
                print(f'  Unique team values (col 2):',   sorted(set(teams_col.tolist())))
        elif isinstance(grp, h5py.Group):
            sub_keys = list(grp.keys())
            print(f'  Sub-keys: {sub_keys[:8]}')
            if sub_keys:
                ds = grp[sub_keys[0]]
                print(f'  [{sub_keys[0]}] shape: {ds.shape}, dtype: {ds.dtype}')
                print(f'  First 3 rows:\n{ds[:3]}')

print('\nDone -- update TACTICAL_CLS_MAP in Cell F based on unique class values above.')

## Cell E - Extract ResNet18 Features
Batch GPU extraction at 2fps. Saves to Drive with TRAIN/VAL prefix so Cell F can split automatically.
Skip-safe: re-running resumes from where it left off.

In [ ]:
import cv2, numpy as np, torch, torch.nn as nn
from pathlib import Path
from torchvision import models, transforms

LOCAL_EXTRACT = Path('/content/pcbas-extracted')
FEAT_DIR      = Path('/content/drive/MyDrive/ezstats/pcbas-features')
FEAT_DIR.mkdir(parents=True, exist_ok=True)
FEAT_FPS   = 2.0
BATCH_SIZE = 512  # A100 40GB VRAM -- ResNet18 224x224 is tiny, push hard

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
backbone.fc = nn.Identity()
backbone = backbone.to(device).eval()
backbone = torch.compile(backbone)

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_features(video_path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    interval = max(1, int(round(fps / FEAT_FPS)))
    frames, idx = [], 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if idx % interval == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(preprocess(rgb))
        idx += 1
    cap.release()
    if not frames:
        return np.zeros((0, 512), dtype=np.float32), fps
    feats = []
    with torch.no_grad():
        for i in range(0, len(frames), BATCH_SIZE):
            batch = torch.stack(frames[i:i+BATCH_SIZE]).to(device)
            with torch.amp.autocast('cuda'):
                out = backbone(batch).float().cpu().numpy()
            norms = np.linalg.norm(out, axis=1, keepdims=True)
            feats.append(out / np.where(norms > 0, norms, 1))
    return np.concatenate(feats).astype(np.float32), fps

videos = sorted(LOCAL_EXTRACT.rglob('*.mp4')) + sorted(LOCAL_EXTRACT.rglob('*.mkv'))
print(f'Found {len(videos)} videos')

for video_path in videos:
    # Use video stem (e.g. 'game_1') as the key -- matches HDF5 game keys in Cell F
    game_id   = video_path.stem
    feat_path = FEAT_DIR / f'{game_id}.npy'
    fps_path  = FEAT_DIR / f'{game_id}.fps.txt'
    if feat_path.exists():
        print(f'  SKIP: {game_id}')
        continue
    print(f'  {game_id}...', end='', flush=True)
    feats, src_fps = extract_features(video_path)
    np.save(str(feat_path), feats)
    fps_path.write_text(str(src_fps))
    print(f' {len(feats)} frames @ {src_fps:.1f}fps')

print('Feature extraction done.')

## Cell F - Build Dataset
Handles both FOOTPASS dict format and SoccerNet annotation list format automatically.
UPDATE: BALL_ACTION_CLASSES if Cell D shows different names. Update get_pos() if positions are ms (>100000 heuristic handles this).

In [ ]:
import h5py, numpy as np
from pathlib import Path
from collections import Counter

LOCAL_EXTRACT = Path('/content/pcbas-extracted')
FEAT_DIR      = Path('/content/drive/MyDrive/ezstats/pcbas-features')
FEAT_FPS      = 2.0
SOURCE_FPS    = 25.0
WINDOW        = 25
RADIUS        = 2

BALL_ACTION_CLASSES = [
    'background',
    'DRIVE',
    'PASS',
    'CROSS',
    'SHOT',
    'HEADER',
    'THROW IN',
    'TACKLE',
    'BLOCK',
]
NUM_CLASSES = len(BALL_ACTION_CLASSES)

# 0=background, 1-8=actions -- confirmed by Cell D. No change needed.
TACTICAL_CLS_MAP = {i: i for i in range(NUM_CLASSES)}

TEAM_NAMES   = ['background', 'left', 'right']
NUM_TEAMS    = 3
TEAM_COL_MAP = {0: 1, 1: 2}

print(f'{NUM_CLASSES} classes:', BALL_ACTION_CLASSES)

def build_samples_from_h5_data(feats, data, window=WINDOW, radius=RADIUS, bg_keep=0.05):
    T          = len(feats)
    act_lbl    = np.zeros(T, dtype=np.int64)
    team_lbl   = np.zeros(T, dtype=np.int64)
    jersey_lbl = np.full(T, -1, dtype=np.int64)

    if data.ndim != 2 or data.shape[1] < 14:
        return []

    frames_col  = data[:, 0].astype(int)
    teams_col   = data[:, 2].astype(int)
    jerseys_col = data[:, 3].astype(int)
    classes_col = np.nan_to_num(data[:, 13], nan=0).astype(int)

    for i in range(len(data)):
        cls = TACTICAL_CLS_MAP.get(int(classes_col[i]), 0)
        if cls == 0: continue
        ff  = int(round(int(frames_col[i]) / SOURCE_FPS * FEAT_FPS))
        tm  = TEAM_COL_MAP.get(int(teams_col[i]), 0)
        jer = int(jerseys_col[i])
        for k in range(max(0, ff - radius), min(T, ff + radius + 1)):
            act_lbl[k]    = cls
            team_lbl[k]   = tm
            jersey_lbl[k] = jer

    samples = []
    for i in range(T - window + 1):
        c = act_lbl[i + window // 2]
        if c == 0 and np.random.rand() > bg_keep: continue
        samples.append((
            feats[i:i+window].copy(),
            int(c),
            int(team_lbl[i + window // 2]),
            int(jersey_lbl[i + window // 2]),
        ))
    return samples

np.random.seed(42)

train_h5 = LOCAL_EXTRACT / 'train_tactical_data.h5'
val_h5   = LOCAL_EXTRACT / 'val_tactical_data.h5'

def collect_from_h5(h5_path, tag):
    samples = []
    if not h5_path.exists():
        print(f'NOT FOUND: {h5_path.name}')
        return samples
    with h5py.File(str(h5_path), 'r') as f:
        keys = sorted(f.keys())
        print(f'[{tag}] {len(keys)} halves in {h5_path.name}')
        for game_id in keys:
            # HDF5 keys are 'game_10_H1'/'game_10_H2' but feature files are 'game_10.npy'
            # Frame indices are absolute -- H2 continues from where H1 ends in the full video
            base_game = game_id.rsplit('_H', 1)[0]
            feat_path = FEAT_DIR / f'{base_game}.npy'
            if not feat_path.exists():
                print(f'  SKIP (no features yet): {game_id} -> {base_game}.npy')
                continue
            feats = np.load(str(feat_path))
            grp   = f[game_id]
            data  = grp[:] if isinstance(grp, h5py.Dataset) else np.concatenate([grp[k][:] for k in sorted(grp.keys())], axis=0)
            s = build_samples_from_h5_data(feats, data)
            samples.extend(s)
            print(f'  [{tag}] {game_id}: {data.shape[0]:>8} rows -> {len(s):>6} samples')
    return samples

train_samples = collect_from_h5(train_h5, 'T')
val_samples   = collect_from_h5(val_h5,   'V')

counts = Counter(s[1] for s in train_samples)
print('\nClass distribution (train):')
for i, name in enumerate(BALL_ACTION_CLASSES):
    if counts[i]: print(f'  {i}  {name:<24} {counts[i]:>7}')
print(f'\nTrain: {len(train_samples)}, Val: {len(val_samples)}')

## Cell G - Train Dual-Head Bi-LSTM
Action head: 9 classes. Team head: left/right (used by Ronan's dashboard later).
Balanced sampler replaces loss weights -- fixes the PASS/DRIVE 0% problem seen in BAS-2025 training.

In [ ]:
import math, torch, torch.nn as nn, numpy as np
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

class EventDataset(Dataset):
    def __init__(self, s, augment=False):
        self.X       = np.array([x[0] for x in s], dtype=np.float32)
        self.act     = [x[1] for x in s]
        self.team    = [x[2] for x in s]
        self.augment = augment
    def __len__(self): return len(self.act)
    def __getitem__(self, i):
        x = self.X[i].copy()
        if self.augment:
            x += np.random.randn(*x.shape).astype(np.float32) * 0.02
        return torch.from_numpy(x), self.act[i], self.team[i]

sample_weights = np.array(
    [1.0 / max(counts[s[1]], 1) for s in train_samples], dtype=np.float32
)
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights),
    num_samples=len(train_samples),
    replacement=True,
)

train_loader = DataLoader(EventDataset(train_samples, augment=True),
                          batch_size=1024, sampler=sampler, num_workers=4, pin_memory=True)
val_loader   = DataLoader(EventDataset(val_samples,   augment=False),
                          batch_size=1024, shuffle=False, num_workers=4, pin_memory=True)

# --- Transformer encoder (replaces Bi-LSTM) ---
# Parallelizes fully on A100 vs LSTM which is sequential.
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64, dropout=0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.drop(x + self.pe[:, :x.shape[1]])

class EventSpotter(nn.Module):
    def __init__(self, n_action, n_team=3, input_dim=512,
                 d_model=512, nhead=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.proj    = nn.Linear(input_dim, d_model)
        self.pe      = PositionalEncoding(d_model, dropout=dropout)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=2048,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder     = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.shared      = nn.Sequential(nn.Linear(d_model, 256), nn.ReLU(), nn.Dropout(dropout))
        self.action_head = nn.Sequential(
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, n_action)
        )
        self.team_head   = nn.Linear(256, n_team)

    def forward(self, x):
        x      = self.pe(self.proj(x))
        out    = self.encoder(x)
        center = out[:, out.shape[1] // 2]
        shared = self.shared(center)
        return self.action_head(shared), self.team_head(shared)

EPOCHS   = 150
WARMUP   = 10  # linear warmup epochs before cosine decay

model          = EventSpotter(NUM_CLASSES, NUM_TEAMS).to(device)
model          = torch.compile(model)  # fused kernels on A100
criterion_act  = nn.CrossEntropyLoss(label_smoothing=0.1)
criterion_team = nn.CrossEntropyLoss()
optimizer      = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def lr_lambda(epoch):
    if epoch < WARMUP:
        return (epoch + 1) / WARMUP
    progress = (epoch - WARMUP) / (EPOCHS - WARMUP)
    return max(1e-6, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.amp.GradScaler('cuda')

best_val_loss, best_state = float('inf'), None

for epoch in range(1, EPOCHS + 1):
    model.train()
    tl = tc = tt = 0
    for X, y_act, y_team in train_loader:
        X, y_act, y_team = X.to(device), y_act.to(device), y_team.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            act_logits, team_logits = model(X)
        act_loss  = criterion_act(act_logits.float(), y_act)
        mask      = y_act != 0
        team_loss = criterion_team(team_logits[mask].float(), y_team[mask]) if mask.any() else act_loss * 0
        loss      = act_loss + 0.3 * team_loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        tl += act_loss.item()*len(y_act)
        tc += (act_logits.argmax(1) == y_act).sum().item()
        tt += len(y_act)

    model.eval()
    vl = vc = vt = 0
    with torch.no_grad():
        for X, y_act, y_team in val_loader:
            X, y_act = X.to(device), y_act.to(device)
            with torch.amp.autocast('cuda'):
                act_logits, _ = model(X)
            loss = criterion_act(act_logits.float(), y_act)
            vl += loss.item()*len(y_act)
            vc += (act_logits.argmax(1) == y_act).sum().item()
            vt += len(y_act)

    scheduler.step()
    flag = ''
    if vl/vt < best_val_loss:
        best_val_loss = vl/vt
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        flag = ' <- best'
    if epoch % 5 == 0 or flag:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'Ep {epoch:03d} lr={lr_now:.2e} | train {tl/tt:.4f} acc={tc/tt:.3f} | val {vl/vt:.4f} acc={vc/vt:.3f}{flag}')

## Cell H - Per-class Accuracy + Prediction Distribution + Team Accuracy

In [ ]:
from collections import defaultdict, Counter

model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
model.eval()
pc, pt = defaultdict(int), defaultdict(int)
all_preds = []
with torch.no_grad():
    for X, y_act, y_team in val_loader:
        X, y_act = X.to(device), y_act.to(device)
        preds = model(X)[0].argmax(1)
        all_preds.extend(preds.cpu().numpy().tolist())
        for t, p in zip(y_act.cpu().numpy(), preds.cpu().numpy()):
            pt[t] += 1
            if t == p: pc[t] += 1

print('Per-class val accuracy (action head):')
for i, name in enumerate(BALL_ACTION_CLASSES):
    if pt[i]:
        bar = '#' * int(pc[i]/pt[i] * 20)
        print(f'  {name:<24} {pc[i]/pt[i]:.2f}  ({pc[i]:>5}/{pt[i]:<5})  {bar}')

print('\nPrediction distribution:')
pred_dist = Counter(all_preds)
total_p   = sum(pred_dist.values())
for i, name in enumerate(BALL_ACTION_CLASSES):
    if pred_dist[i]:
        print(f'  {name:<24} {pred_dist[i]:>6}  ({pred_dist[i]/total_p*100:.1f}%)')

tc_team = tt_team = 0
with torch.no_grad():
    for X, y_act, y_team in val_loader:
        X, y_act, y_team = X.to(device), y_act.to(device), y_team.to(device)
        _, team_logits = model(X)
        mask = y_act != 0
        if mask.any():
            tc_team += (team_logits[mask].argmax(1) == y_team[mask]).sum().item()
            tt_team += mask.sum().item()
if tt_team:
    print(f'\nTeam accuracy (event frames only): {tc_team/tt_team:.3f} ({tc_team}/{tt_team})')

## Cell I - Save Model to Drive

In [ ]:
import json, torch
from pathlib import Path

save_dir = Path('/content/drive/MyDrive/ezstats/runs/event_spotter_pcbas2026')
save_dir.mkdir(parents=True, exist_ok=True)
torch.save(best_state, str(save_dir / 'model.pt'))
(save_dir / 'classes.json').write_text(json.dumps(BALL_ACTION_CLASSES, indent=2))
(save_dir / 'team_classes.json').write_text(json.dumps(TEAM_NAMES, indent=2))
print(f'Saved -> {save_dir}/model.pt')
print('Next steps on laptop:')
print('  1. Download model.pt -> artifacts/training/event_spotter_pcbas2026/model.pt')
print('  2. Update event_spotter.py: 9 classes, dual-head (action+team), head output 18->9')
print('  3. Update run scripts: --event-model-name artifacts/training/event_spotter_pcbas2026/model.pt')